# 第 6 章　随机数、排序与高级技巧

> 📖 **本章目标**
>
> - 彻底告别旧式随机 API，掌握现代 `default_rng` + Generator 对象
> - 学会 Generator 的常用方法：uniform / integers / normal / choice / shuffle / permutation
> - 掌握可复现性实践：固定 seed 让结果可复现
> - 精通排序：np.sort / argsort / lexsort / partition 的取舍
> - 掌握唯一值与集合运算、常用搜索函数
> - 学会文件 IO：.npy 二进制 vs .csv 文本的取舍
> - 了解性能与内存最佳实践、结构化数组
> - 最后用一张 6 章知识地图收尾整个系列

> 🔗 这是系列的第 6 章，也是最后一章。前五章我们走完了：ndarray 基础与 dtype（第 1 章）、索引切片与布尔筛选（第 2 章）、向量化运算与广播（第 3 章）、形状变换与拼接（第 4 章）、统计聚合与线性代数（第 5 章）。本章把这些工具拼成“能落地的完整武器库”。

In [1]:
import numpy as np
print(np.__version__)

2.4.4


## 6.1　现代随机 API：default_rng 与 Generator 对象

### 先看问题：旧式 API 有什么毛病？

老代码里最常见的写法是 `np.random.seed(0)` + `np.random.rand(...)`。它有几个天生缺陷：

- **全局状态**：`seed` 设置的是“整个模块的全局随机状态”，谁调用都会影响谁，多线程 / 多函数里很难隔离；
- **不便于并行**：想并行生成多路随机流，旧式 API 很难做到互不干扰；
- **命名混乱**：`rand`（均匀分布）、`randn`（正态分布）、`randint`（整数）三兄弟长得像但语义不同，容易用错。

### 新式 API 的思路

新式 API 的核心是 **`Generator` 对象**：你把它当“私人的随机数生成器”，方法和随机状态都挂在它自己身上，互不干扰。

| 维度 | 旧式（不推荐） | 新式（推荐） |
|------|----------------|--------------|
| 创建 | `np.random.seed(0)` 全局设置 | `rng = np.random.default_rng(0)` |
| 调用 | `np.random.rand(3)` 模块级函数 | `rng.random(3)` 对象方法 |
| 状态 | 全局、隐式、难隔离 | 对象内、显式、可复制 |
| 并行 | 困难 | 每个 Generator 独立，天然可并行 |
| 官方态度 | 仅为了兼容旧代码而保留 | **NumPy 2.x 明确推荐** |

> 💡 NumPy 官方文档原话大意：`default_rng` 是“新的正道”（the recommended way），旧式 `np.random.seed/rand/randint` 只是为了向后兼容。新项目请一律使用新式 API。

```mermaid
flowchart LR
    subgraph old["旧式：全局共享状态"]
        SEED["np.random.seed(0)<br/>设置全局状态"] --> G1["np.random.rand()"]
        SEED --> G2["np.random.randn()"]
        SEED --> G3["np.random.randint()"]
        G1 -.互相影响.-> G2
        G2 -.互相影响.-> G3
    end
    subgraph new["新式：独立 Generator"]
        R1["rng1 = default_rng(1)"] --> A1["rng1.random()"]
        R2["rng2 = default_rng(2)"] --> B1["rng2.normal()"]
        R1 -.互不影响.-> R2
    end
```

In [2]:
# 新式 API：创建独立 Generator，传入 seed 保证可复现
rng = np.random.default_rng(42)

print('rng.random(5):', rng.random(5))   # 均匀分布 [0,1)
print('rng.random():', rng.random())     # 单个标量

# 用同一个 seed 重建，序列完全一致（可复现）
rng_a = np.random.default_rng(7)
rng_b = np.random.default_rng(7)
print('同一 seed 序列一致:', np.array_equal(rng_a.random(5), rng_b.random(5)))

rng.random(5): [0.77395605 0.43887844 0.85859792 0.69736803 0.09417735]
rng.random(): 0.9756223516367559
同一 seed 序列一致: True


## 6.2　Generator 常用方法

### 连续分布：random / uniform / normal

| 方法 | 分布 | 说明 |
|------|------|------|
| `rng.random(size)` | 均匀分布 [0,1) | 最基础 |
| `rng.uniform(lo, hi, size)` | 均匀分布 [lo, hi) | 指定范围 |
| `rng.normal(mu, sigma, size)` | 正态分布 | 默认 mu=0, sigma=1 |
| `rng.standard_normal(size)` | 标准正态 N(0,1) | 等价 normal(0,1) |

In [3]:
rng = np.random.default_rng(1)

print('random(4):', rng.random(4))
print('uniform(0, 10, 4):', np.round(rng.uniform(0, 10, 4), 3))
print('normal(mean=5, std=2, 4):', np.round(rng.normal(5, 2, 4), 3))
print('standard_normal(4):', np.round(rng.standard_normal(4), 3))

random(4): [0.51182162 0.9504637  0.14415961 0.94864945]
uniform(0, 10, 4): [3.118 4.233 8.277 4.092]
normal(mean=5, std=2, 4): [5.729 5.588 5.057 6.093]
standard_normal(4): [-0.736 -0.163 -0.482  0.599]


### 整数：integers（注意 endpoint 参数！）

`rng.integers(low, high, size)` 默认是 **[low, high)** 左闭右开，即**取不到 high**。这是最容易踩的坑：

| 写法 | 取值范围 |
|------|----------|
| `rng.integers(0, 10)` | 0 ~ 9（默认不含 10） |
| `rng.integers(0, 10, endpoint=True)` | 0 ~ 10（含 10） |

In [4]:
rng = np.random.default_rng(2)

print('integers(0,10) 范围 0~9:', rng.integers(0, 10, 10))
print('integers(1,7) 掷骰子(1~6):', rng.integers(1, 7, 10))
print('integers(0,10,endpoint=True) 含10:', rng.integers(0, 10, 10, endpoint=True))

integers(0,10) 范围 0~9: [8 2 1 2 4 8 4 0 3 6]
integers(1,7) 掷骰子(1~6): [5 5 6 2 6 1 4 2 2 4]
integers(0,10,endpoint=True) 含10: [ 3  6  2  1  8  4  7  7 10  4]


### 抽样：choice（是否放回、权重 p）

`rng.choice(a, size, replace, p)` 从数组 `a` 中抽样：

- `replace=True`（默认）：**放回**抽样，可重复抽到同一项
- `replace=False`：**不放回**抽样，每一项最多出现一次（size 不能超过 a 的长度）
- `p`：给每个元素指定被抽中的概率（权重）

In [5]:
rng = np.random.default_rng(3)

pool = np.array([10, 20, 30, 40, 50])
print('不放回抽 3 个:', rng.choice(pool, size=3, replace=False))
print('放回抽 8 个:', rng.choice(pool, size=8, replace=True))

# 带权重的抽样：比如抽“中奖者”，一等奖权重更高
items = np.array(['普通', '优秀', '特等奖'])
p = np.array([0.7, 0.25, 0.05])
print('按权重抽 10 次:', rng.choice(items, size=10, p=p))

不放回抽 3 个: [10 50 30]
放回抽 8 个: [50 50 30 10 10 20 30 40]
按权重抽 10 次: ['普通' '优秀' '普通' '普通' '普通' '普通' '普通' '优秀' '特等奖' '普通']


### shuffle（原地）vs permutation（返回拷贝）

这两个都用于“打乱顺序”，但**一个原地改、一个返回新数组**：

| 方法 | 是否原地 | 是否返回 | 使用建议 |
|------|----------|----------|----------|
| `rng.shuffle(a)` | 是，直接改 `a` | 返回 None | 确定要改原数组 |
| `rng.permutation(a)` | 否，`a` 不变 | 返回打乱后的新数组 | 想保留原数组 |

> ⚠️ **陷阱**：`rng.shuffle` 返回 `None`！新手常写 `b = rng.shuffle(a)` 结果 b 是 None。

In [6]:
rng = np.random.default_rng(4)
a = np.arange(6)
b = a.copy()

print('shuffle 前 a:', a)
rng.shuffle(a)                      # 原地打乱
print('shuffle 后 a:', a)

print()
print('permutation 前 b:', b)
c = rng.permutation(b)              # b 不变，c 是新的
print('permutation 后 b:', b)
print('permutation 返回 c:', c)

shuffle 前 a: [0 1 2 3 4 5]
shuffle 后 a: [1 2 0 5 4 3]

permutation 前 b: [0 1 2 3 4 5]
permutation 后 b: [0 1 2 3 4 5]
permutation 返回 c: [1 0 4 2 5 3]


### standard 系列分布速览

Generator 还内置了一批 `standard_*` 方法，都是“标准参数化”的常用分布，做蒙特卡洛模拟很方便：

| 方法 | 分布 | 一句话用途 |
|------|------|------------|
| `standard_normal` | 标准正态 | 通用误差 / 噪声 |
| `standard_exponential` | 指数分布 | 等待时间、泊松过程 |
| `standard_gamma(shape)` | Gamma 分布 | 广义指数 / 排队 |
| `standard_cauchy` | 柯西分布 | 重尾分布例子 |
| `standard_t(df)` | t 分布 | 小样本统计推断 |

更完整的分布（beta、poisson、binomial、multinomial……）都在 `rng.` 上：`dir(rng)` 能看到全貌。

In [7]:
rng = np.random.default_rng(5)
print('standard_exponential(5):', np.round(rng.standard_exponential(5), 3))
print('standard_gamma(shape=2, 5):', np.round(rng.standard_gamma(2.0, 5), 3))
print('standard_cauchy(5):', np.round(rng.standard_cauchy(5), 3))
print('standard_t(df=10, 5):', np.round(rng.standard_t(10, 5), 3))

standard_exponential(5): [1.987 0.75  1.301 0.529 0.03 ]
standard_gamma(shape=2, 5): [1.812 0.845 4.793 0.528 4.703]
standard_cauchy(5): [20.696  1.848  0.684 -8.772 -1.439]
standard_t(df=10, 5): [ 1.332 -1.057  0.022 -1.192 -1.333]


## 6.3　可复现性实践

做实验、跑论文复现、调随机参数，**必须让随机结果可复现**。方法很简单：**传固定 seed**。

```mermaid
flowchart LR
    Seed["seed = 42"] --> BC["BitGenerator (PCG64)"] --> SS["SeedSequence<br/>把seed拆成独立子流"] --> G["rng = default_rng(seed)"]
```

> 💡 一句话概念：`default_rng(seed)` 内部先用 `SeedSequence` 把种子（整数或字节）转化成确定性序列，再驱动底层的 `BitGenerator`（默认 PCG64）产生随机数。你只需要记住：**同一个 seed → 同一串随机数**。

In [8]:
# 演示：固定 seed 的可复现性
def roll_some(seed):
    rng = np.random.default_rng(seed)
    return rng.integers(1, 7, 5)

print('第一次调用:', roll_some(2024))
print('第二次调用:', roll_some(2024))     # 完全相同
print('换种子:', roll_some(999))          # 不同

# 实用技巧：同一 rng 按顺序调用，序列是确定的
rng = np.random.default_rng(0)
print()
print('前两次调用:', rng.random(2))
print('第三次调用:', rng.random(3))

第一次调用: [2 5 1 2 2]
第二次调用: [2 5 1 2 2]
换种子: [5 5 2 2 2]

前两次调用: [0.63696169 0.26978671]
第三次调用: [0.04097352 0.01652764 0.81327024]


## 6.4　排序

### np.sort（返回拷贝）vs arr.sort()（原地）

| 写法 | 是否原地 | 返回 | 使用场景 |
|------|----------|------|----------|
| `np.sort(a)` | 否 | 排序后的新数组 | 想保留原数组 |
| `a.sort()` | 是 | None | 确定要改原数组 |
| `np.sort(a, axis=k)` | 否 | 沿第 k 轴排序的新数组 | 多维排序 |

> 💡 默认 `axis=-1`，即对**最后一个轴**排序。二维数组默认就是“逐行排序”。

In [9]:
a = np.array([3, 1, 4, 1, 5])
b = np.sort(a)
print('np.sort(a):', b, '  原数组 a 不变:', a)

a.sort()                              # 原地排序
print('a.sort() 后 a:', a)

# 二维数组按 axis 排序
M = np.array([[3, 1, 2],
              [9, 7, 8]])
print()
print('axis=0 排序（按列）:')
print(np.sort(M, axis=0))
print('axis=1 排序（按行，默认）:')
print(np.sort(M, axis=1))

np.sort(a): [1 1 3 4 5]   原数组 a 不变: [3 1 4 1 5]
a.sort() 后 a: [1 1 3 4 5]

axis=0 排序（按列）:
[[3 1 2]
 [9 7 8]]
axis=1 排序（按行，默认）:
[[1 2 3]
 [7 8 9]]


### argsort：间接排序（排的是“名次”，不是数据本身）

`argsort` 返回的**不是排序后的值，而是排序后各元素在原数组中的下标**。它最大的用处：**按一个数组排序，同时让另一个数组跟着走**（比如按成绩排名并还原姓名）。

In [10]:
scores = np.array([88, 55, 92, 70])
names  = np.array(['张三', '李四', '王五', '赵六'])

order = np.argsort(scores)             # 升序的名次（下标）
print('argsort 下标:', order)          # 分数升序：55,70,88,92 -> 李四,赵六,张三,王五

print()
print('升序排名:')
for rank, i in enumerate(order, 1):
    print(f'  第{rank}名: {names[i]}  分数 {scores[i]}')

# 想按分数从高到低排 -> 反转或取负
desc = np.argsort(-scores)
print()
print('降序(高分在前):', names[desc])

argsort 下标: [1 3 0 2]

升序排名:
  第1名: 李四  分数 55
  第2名: 赵六  分数 70
  第3名: 张三  分数 88
  第4名: 王五  分数 92

降序(高分在前): ['王五' '张三' '赵六' '李四']


### lexsort：多关键字排序

当排序有**多个依据**（先按班级、再按分数）时用 `lexsort`。**关键点：传入的键从最后一个开始作为“最重要的关键字”**。

> 💡 记忆：`np.lexsort((次要键, 主要键))` —— 和直觉相反，最后面的键优先级最高。

In [11]:
classes = np.array(['二班', '一班', '二班', '一班', '一班'])
scores  = np.array([70, 90, 85, 60, 95])
names   = np.array(['甲', '乙', '丙', '丁', '戊'])

# 先按班级（主要），班级内按分数（次要）: lexsort((分数, 班级))
order = np.lexsort((scores, classes))
print('排序后的 班级-姓名-分数:')
for i in order:
    print(f'  {classes[i]}  {names[i]}  {scores[i]}')

排序后的 班级-姓名-分数:
  一班  丁  60
  一班  乙  90
  一班  戊  95
  二班  甲  70
  二班  丙  85


### partition / argpartition：部分排序（找 top-K 不用全排序）

如果只想找“最小的 K 个”，`np.partition(a, k)` 比 `np.sort` **快得多**：它只保证第 k 个位置放的是“第 k+1 小”的值，左侧都 ≤ 它、右侧都 ≥ 它，两侧内部**不保证有序**。

| 函数 | 返回 | 复杂度 | 场景 |
|------|------|--------|------|
| `np.sort` | 完整有序 | O(n log n) | 需要全序 |
| `np.partition` | 第 k 位就位，两边乱序 | O(n) | 只要 top-K |
| `np.argpartition` | top-K 的**下标** | O(n) | 只要 top-K 的位置 |

In [12]:
rng = np.random.default_rng(6)
big = rng.integers(0, 1000, size=100_000)

# 只要最小的 5 个：partition 只排一部分，快很多
small_vals = np.partition(big, 4)[:5]
print('最小的5个(顺序不一定):', small_vals)

# 或者只要下标
idx = np.argpartition(big, 4)[:5]
print('最小的5个的下标:', idx)
print('验证(用这些下标取值):', big[idx])

最小的5个(顺序不一定): [0 0 0 0 0]
最小的5个的下标: [67706 51079 27070 43214 63520]
验证(用这些下标取值): [0 0 0 0 0]


## 6.5　唯一值与集合运算

### np.unique：去重 + 三个“副产品”

`np.unique` 不只是去重，还能用参数**顺便**返回额外信息，省一次遍历：

| 参数 | 返回内容 |
|------|----------|
| `return_index=True` | 每个唯一值**第一次出现**的下标 |
| `return_counts=True` | 每个唯一值**出现的次数**（词频统计！） |
| `return_inverse=True` | 原数组每个元素 → 唯一值数组的哪个下标（用于“重建/反查”） |

In [13]:
scores = np.array([88, 55, 88, 92, 55, 55, 100, 92])

uniq = np.unique(scores)
print('唯一值(自动排序):', uniq)

# return_counts：词频统计（分数 -> 出现次数）
vals, counts = np.unique(scores, return_counts=True)
print('分数:', vals)
print('次数:', counts)

# return_index：首次出现位置
vals, idx = np.unique(scores, return_index=True)
print('唯一值首次出现下标:', idx)

# return_inverse：重建反查 —— 用唯一值把原数组“压缩”表示
vals, inv = np.unique(scores, return_inverse=True)
print('inverse(每个元素在唯一值里的下标):', inv)
print('用 vals[inv] 还原原数组:', vals[inv])

唯一值(自动排序): [ 55  88  92 100]
分数: [ 55  88  92 100]
次数: [3 2 2 1]
唯一值首次出现下标: [1 0 3 6]
inverse(每个元素在唯一值里的下标): [1 0 1 2 0 0 3 2]
用 vals[inv] 还原原数组: [ 88  55  88  92  55  55 100  92]


### 集合运算

两个数组做集合（交集、并集、差集……）：

| 函数 | 含义 |
|------|------|
| `np.intersect1d(a, b)` | 交集（同时出现在 a 和 b） |
| `np.union1d(a, b)` | 并集 |
| `np.setdiff1d(a, b)` | 差集（在 a 但不在 b） |
| `np.setxor1d(a, b)` | 对称差（只在一边出现） |
| `np.isin(a, b)` | 逐元素判断 a 的每个值是否在 b 中（返回布尔数组） |

> 💡 `np.isin` 是最常用的：比如“这批订单里，哪些属于 VIP 客户名单”。

In [14]:
a = np.array([1, 2, 3, 4, 5])
b = np.array([4, 5, 6, 7])

print('交集 intersect1d:', np.intersect1d(a, b))
print('并集 union1d:', np.union1d(a, b))
print('差集 setdiff1d(a,b):', np.setdiff1d(a, b))   # 在a不在b
print('对称差 setxor1d:', np.setxor1d(a, b))
print('isin 判断 a 中哪些值在 b:', np.isin(a, b))

交集 intersect1d: [4 5]
并集 union1d: [1 2 3 4 5 6 7]
差集 setdiff1d(a,b): [1 2 3]
对称差 setxor1d: [1 2 3 6 7]
isin 判断 a 中哪些值在 b: [False False False  True  True]


## 6.6　搜索：找位置、找下标、找区间

### argmax / argmin / nonzero：回顾 + 定位

第 2 章我们用过布尔筛选，这里补充“**定位**”三件套：

- `argmax / argmin`：最值下标（第 5 章讲过，复习一下）
- `np.nonzero(cond)`：返回**满足条件的位置坐标**
- `np.where(cond)`：等价于 nonzero，返回坐标元组

In [15]:
rng = np.random.default_rng(7)
data = rng.integers(0, 20, 10)
print('data:', data)
print('最大值:', data.max(), '位置 argmax:', data.argmax())

# nonzero：找出所有“偶数”的位置
even_mask = data % 2 == 0
positions = np.nonzero(even_mask)
print('偶数值:', data[even_mask])
print('偶数下标:', positions[0])

# 二维数组：nonzero 返回 (行下标, 列下标)
M = np.array([[0, 5, 0],
              [7, 0, 9]])
rows, cols = np.nonzero(M)
print()
print('M 中非零元素坐标:', list(zip(rows, cols)))

data: [18 12 13 17 11 15 16  4  1  6]
最大值: 18 位置 argmax: 0
偶数值: [18 12 16  4  6]
偶数下标: [0 1 6 7 9]

M 中非零元素坐标: [(np.int64(0), np.int64(1)), (np.int64(1), np.int64(0)), (np.int64(1), np.int64(2))]


### searchsorted：二分查找插入位置（要求已排序！）

`searchsorted` 用**二分法**在**已排序**数组里找“某个值该插入的位置”，比线性查找快。常用于：给新数据找它应该落在哪个区间、合并多个有序数组。

> ⚠️ **陷阱**：输入必须已排序，否则结果无意义。

In [16]:
sorted_arr = np.array([10, 20, 30, 40, 50])
print('已排序:', sorted_arr)
print('30 应该插在?', np.searchsorted(sorted_arr, 30))
print('35 应该插在?', np.searchsorted(sorted_arr, 35))
print('5 应该插在?', np.searchsorted(sorted_arr, 5))
print('60 应该插在?', np.searchsorted(sorted_arr, 60))

# 批量查询
new_vals = np.array([25, 45, 15])
print('批量:', np.searchsorted(sorted_arr, new_vals))

已排序: [10 20 30 40 50]
30 应该插在? 2
35 应该插在? 3
5 应该插在? 0
60 应该插在? 5
批量: [2 4 1]


### digitize：分箱（按分数段评等级）

`digitize(x, bins)` 把连续数值**分到离散的箱**里，返回每个值所属箱的编号。做“分段统计”“评级”时一绝。

In [17]:
scores = np.array([58, 73, 65, 91, 40, 88, 60])
bins   = np.array([0, 60, 70, 80, 90])        # 分界点

levels = np.digitize(scores, bins)
print('分箱结果:', levels)

grade_names = ['不及格', '及格', '中等', '良好', '优秀']
grades = [grade_names[i - 1] for i in levels]
print('分数:', scores)
print('评级:', grades)

分箱结果: [1 3 2 5 1 4 2]
分数: [58 73 65 91 40 88 60]
评级: ['不及格', '中等', '及格', '优秀', '不及格', '良好', '及格']


### extract：条件提取

`np.extract(cond, a)` 等价于 `a[cond]`，纯粹是函数式写法，有时在链式处理里更好读。

In [18]:
a = np.array([3, -1, 5, -2, 8, -4])
print('np.extract(a > 0, a):', np.extract(a > 0, a))   # 等价于 a[a>0]
print('等价写法 a[a>0]:', a[a > 0])

np.extract(a > 0, a): [3 5 8]
等价写法 a[a>0]: [3 5 8]


## 6.7　文件 IO：保存与加载

NumPy 提供两种文件格式，用途完全不同：

| 维度 | `.npy`（二进制） | `.csv`（文本） |
|------|------------------|----------------|
| 内容 | 纯 NumPy 数组，含 dtype/shape | 通用文本，Excel 能开 |
| 体积 | **小**（二进制紧凑存储） | 大（文本冗余） |
| 读写速度 | **快** | 慢 |
| 保真度 | 完全保留 dtype、shape、NaN、inf | 可能丢精度、丢失类型信息 |
| 互操作 | 只有 NumPy | 任何工具 |

> 💡 **原则**：程序内部、中间结果用 `.npy`；要给别人 / 其他软件看、或给 Excel 用，再导出 `.csv`。

In [19]:
import os
os.makedirs('tmp_io', exist_ok=True)

# 二进制 .npy：保留 dtype 和 shape
arr = np.array([[1.5, 2.5, 3.5],
                [4.5, 5.5, 6.5]], dtype=np.float32)
np.save('tmp_io/arr.npy', arr)

loaded = np.load('tmp_io/arr.npy')
print('加载后 dtype:', loaded.dtype, ' shape:', loaded.shape)
print('内容:', loaded)

加载后 dtype: float32  shape: (2, 3)
内容: [[1.5 2.5 3.5]
 [4.5 5.5 6.5]]


In [20]:
# 多数组打包：savez / savez_compressed
a = np.arange(5)
b = np.linspace(0, 1, 5)
np.savez('tmp_io/multi.npz', a=a, b=b)
np.savez_compressed('tmp_io/multi_comp.npz', a=a, b=b)   # 压缩版，体积更小

data = np.load('tmp_io/multi.npz')
print('npz 里的键:', list(data.keys()))
print('a:', data['a'], ' b:', data['b'])

npz 里的键: ['a', 'b']
a: [0 1 2 3 4]  b: [0.   0.25 0.5  0.75 1.  ]


In [21]:
# 文本 .csv：savetxt / loadtxt（可用 fmt 控制精度）
x = np.array([[1.5, 2.0],
              [3.5, 4.0]])
np.savetxt('tmp_io/data.csv', x, delimiter=',', fmt='%.2f')
with open('tmp_io/data.csv') as f:
    print(f.read())

back = np.loadtxt('tmp_io/data.csv', delimiter=',')
print('loadtxt 读回:')
print(back)

1.50,2.00
3.50,4.00

loadtxt 读回:
[[1.5 2. ]
 [3.5 4. ]]


In [22]:
# genfromtxt：处理缺失值（空字段自动变 NaN）
with open('tmp_io/missing.csv', 'w') as f:
    print('1,2,3', file=f)
    print('4,,6', file=f)
    print('7,8,', file=f)

m = np.genfromtxt('tmp_io/missing.csv', delimiter=',')
print('genfromtxt 读回(空值→NaN):')
print(m)
print('nanmean 每行均值:', np.nanmean(m, axis=1))

genfromtxt 读回(空值→NaN):
[[ 1.  2.  3.]
 [ 4. nan  6.]
 [ 7.  8. nan]]
nanmean 每行均值: [2.  5.  7.5]


### 临时文件说明

上面例子写入的 `.npy / .npz / .csv` 都保存在当前目录的 `tmp_io/` 子文件夹里，作为教学示例可以保留。真实项目里，`.npy` 通常存在中间缓存目录，`.csv` 用于交付。

## 6.8　性能与内存最佳实践清单

> 📋 **清单**（第 3 章我们已实践过前两条，这里系统化）

1. **优先向量化**：能用 `numpy` 函数解决就绝不用 Python 循环（差几十到上百倍）；
2. **优先视图而非拷贝**：切片、`reshape`、`transpose` 默认是视图，零拷贝、零耗时；确认要独立数据才用 `.copy()`；
3. **预分配而非循环 append**：`np.append` / Python `list.append` 反复调用很慢，先 `np.empty(n)` 再填；
4. **连续内存更快**：`C order`（行优先）是默认，遍历按行更快；转置视图会变成非连续，重复访问可能变慢；
5. **复杂运算用 einsum**：`np.einsum` 用爱因斯坦记法一步写出多维求和 / 变换，既快又不容易错。

下面用计时实验验证第 1、3 两条。

In [23]:
import time

# ① 向量化 vs Python 循环（同一个求和任务）
n = 1_000_000
big = np.arange(n, dtype=np.float64)

t0 = time.perf_counter()
s_loop = sum(big[i] for i in range(n))          # Python 循环
t_loop = time.perf_counter() - t0

t0 = time.perf_counter()
s_vec = big.sum()                               # 向量化
t_vec = time.perf_counter() - t0

print(f'循环: {t_loop:.3f}s   向量化: {t_vec*1000:.2f}ms   加速比 ≈ {t_loop/t_vec:.0f}x')
print('结果一致:', np.isclose(s_loop, s_vec))

循环: 0.091s   向量化: 0.71ms   加速比 ≈ 127x


结果一致: True


In [24]:
# ② 预分配 vs 循环 append
n = 20_000
t0 = time.perf_counter()
out_append = np.array([])
for i in range(n):                              # 反复 append（很慢！）
    out_append = np.append(out_append, i)
t_append = time.perf_counter() - t0

t0 = time.perf_counter()
out_pre = np.empty(n)                           # 预分配后填
for i in range(n):
    out_pre[i] = i
t_pre = time.perf_counter() - t0

print(f'np.append 循环: {t_append:.2f}s   预分配填充: {t_pre:.3f}s')
print('结果一致:', np.array_equal(out_append, out_pre))

np.append 循环: 0.06s   预分配填充: 0.001s


结果一致: True


In [25]:
# ③ 视图 vs 拷贝：视图零开销
A = np.arange(10000).reshape(100, 100)
view = A[:50, :]           # 视图：不复制数据
copy = A[:50, :].copy()    # 拷贝
view[0, 0] = -999
print('视图修改会改到原数组 A[0,0]:', A[0, 0])
print('拷贝不受影响 copy[0,0]:', copy[0, 0])
print('视图与拷贝是否共享内存:', np.shares_memory(view, A))

# ④ einsum：一行完成矩阵乘 + 对角线求和
X = np.arange(9.0).reshape(3, 3)
Y = X + 1
print()
print('einsum 矩阵乘:', np.allclose(np.einsum('ij,jk->ik', X, Y), X @ Y))
print('einsum 对角线之和(迹):', np.einsum('ii->', X))

视图修改会改到原数组 A[0,0]: -999
拷贝不受影响 copy[0,0]: 0
视图与拷贝是否共享内存: True

einsum 矩阵乘: True
einsum 对角线之和(迹): 12.0


## 6.9　结构化数组（record array）简述

结构化数组是“带字段名的数组”：每个元素不是单个数值，而是一组**命名**字段。它有点像结构体的数组，也是 pandas DataFrame 的雏形。

> 💡 **什么时候用？** 一般数据表格建议直接用 pandas。结构化数组作为知识了解即可：当你在纯 NumPy 环境、不想引入 pandas，又需要“字段名 + 批量向量化”时，它能顶上。

| 维度 | 结构化数组 | pandas DataFrame |
|------|------------|------------------|
| 定位 | NumPy 内置，轻量 | 数据分析标配 |
| 字段访问 | `arr["字段"]` | `df["列"]` |
| 功能 | 基础向量化 | 分组、透视、时间序列…… |

In [26]:
# 定义带字段名的 dtype
dtype_student = np.dtype([('name', 'U10'), ('class', 'U4'), ('score', 'f8')])

students = np.array([
    ('张三', '一班', 88.5),
    ('李四', '二班', 76.0),
    ('王五', '一班', 93.0),
], dtype=dtype_student)

print('整个表:')
print(students)
print()
print('按字段取一列:', students['name'])
print('按字段取一列:', students['score'])

# 字段可以与向量化结合：按成绩筛人
top = students[students['score'] >= 80]
print()
print('成绩 >= 80 的同学:', top['name'])

整个表:
[('张三', '一班', 88.5) ('李四', '二班', 76. ) ('王五', '一班', 93. )]

按字段取一列: ['张三' '李四' '王五']
按字段取一列: [88.5 76.  93. ]

成绩 >= 80 的同学: ['张三' '王五']


## 6.10　全系列回顾：6 章知识地图

从第 1 章的 `ndarray` 到第 6 章的高级技巧，你其实已经走完了一条完整的 NumPy 学习路径：

```mermaid
mindmap
  root((NumPy 六章知识地图))
    第1章 基础
      ndarray 创建
      dtype 数据类型
      shape 形状
    第2章 索引
      切片
      布尔筛选
      fancy index
    第3章 向量化
      通用函数 ufunc
      广播 broadcast
    第4章 形状
      reshape
      拼接 concatenate
      拆分 split
    第5章 统计与线性代数
      聚合 axis
      NaN 处理
      linalg 矩阵运算
    第6章 高级技巧
      随机数 default_rng
      排序与搜索
      文件 IO
      结构化数组
```

### 系列速查

| 章 | 一句话能力 |
|----|-----------|
| 1 | 创建数组、理解 dtype/shape |
| 2 | 索引、切片、布尔筛选 |
| 3 | 向量化运算、广播 |
| 4 | reshape / concatenate / split |
| 5 | 聚合、NaN、线性代数 |
| 6 | 随机数、排序、IO、结构化数组 |

掌握这六章，你已经具备用 NumPy 处理绝大多数数值计算任务的能力——接下来无论是 pandas、matplotlib 还是深度学习框架，底层都是这些概念。

## 6.11　本章小结

### 一句话记忆表

| 主题 | 一句话记忆 |
|------|-----------|
| 随机 API | 用 `rng = np.random.default_rng(seed)`，别再用 `np.random.seed` |
| 可复现 | 固定 seed → 同一串随机数；`integers` 默认不含右端点 |
| shuffle vs permutation | `shuffle` 原地改、`permutation` 返回新数组 |
| 排序 | `np.sort` 返回新、`arr.sort()` 原地；只要 top-K 用 `partition` |
| 排名次 | 用 `argsort` 拿到下标，再让姓名数组跟着走 |
| 集合 | `intersect1d / union1d / setdiff1d / setxor1d / isin` |
| 搜索 | `searchsorted` 需已排序（二分）；`digitize` 分箱 |
| 文件 | 内部用 `.npy`（小、快、保真），交付用 `.csv` |
| 性能 | 向量化 > 循环、视图 > 拷贝、预分配 > append |

### 📝 综合练习（3 题）

1. 用 `default_rng(42)` 生成 100 个标准正态数，排序后取出最大的 5 个（用 `partition` 和 `sort` 各做一次并比较结果是否一致）。
2. 用 `np.unique(return_counts=True)` 统计字符串数组 `['苹果', '香蕉', '苹果', '橙子', '香蕉', '苹果']` 的词频，并打印出出现次数最多的水果。
3. 生成 10 个 0~100 的随机分数，用 `digitize` 分成“不及格/及格/良好/优秀”四档，统计每档人数。

### 全系列结语

> 🎓 **恭喜你完成整个 NumPy 系列教程！** 从 `ndarray` 的一砖一瓦，到聚合、线性代数、随机数、排序与 IO，你已经掌握了 NumPy 的完整武器库。接下来的学习路线建议：
>
> 1. **pandas**：表格数据分析的下一站（结构化数组正是它的雏形）；
> 2. **matplotlib**：把数组变成可视化；
> 3. **SciPy / scikit-learn / PyTorch**：科学计算与机器学习，底层都建立在 NumPy 之上。
>
> 记住那句话：**一切高级库，都是 NumPy 的化妆师。** 基础越牢，走得越远。🚀